# AI in Healthcare — analyse des exécutions
Lot de test multi-agents (6 secteurs) : KPI, Correlation ID, alertes, incident.

*Exécuter depuis la racine du dépôt (`jupyter lab` ou VS Code).*

In [ ]:
import sys, json, random
sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
from app import config, governance, pipeline, stats
cfg = governance.config_active()
print('Version active :', cfg['version'], '| seuil HITL :', cfg['seuil_hitl'])

## 1. Exécuter le lot de test

In [ ]:
demandes = pipeline.charger_lot(config.DATA / 'lot_demandes.json')
rng = random.Random(42)
res = [pipeline.executer(d['secteur'], d['entite_id'], d['indicateurs'], cfg, rng)[0] for d in demandes]
df = pd.DataFrame(res)
df[['correlation_id','secteur','entite_id','score','categorie','route','statut','latence_ms']].head(12)

## 2. Critère d'observabilité : Correlation ID sur chaque exécution

In [ ]:
assert df['correlation_id'].str.match(r'^CID-[0-9a-f]{10}$').all()
assert df['correlation_id'].is_unique
print(f"{len(df)} exécutions, {df['correlation_id'].nunique()} Correlation ID uniques : OK")

## 3. KPI du monitoring

In [ ]:
k = stats.kpi(res)
pd.Series(k).to_frame('valeur')

## 4. Graphiques

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
df['categorie'].replace('', 'Échec').value_counts().plot.bar(ax=ax[0], color=['#2A9D8F','#E9A100','#C0392B','#5B6F75'][:df['categorie'].replace('', 'Échec').nunique()], title='Catégories de risque')
df['secteur'].value_counts().plot.barh(ax=ax[1], color='#0F4C5C', title='Exécutions par secteur')
ax[2].plot(df['latence_ms'].values, marker='o', color='#0F4C5C'); ax[2].axhline(5000, color='#C0392B', ls='--'); ax[2].set_title('Latence (ms), seuil 5 000')
plt.tight_layout(); plt.show()

## 5. Santé : distribution des scores de triage

In [ ]:
sante = df[(df.secteur == 'sante') & (df.score != '')].copy(); sante['score'] = sante['score'].astype(int)
sante[['entite_id','score','categorie','route']].sort_values('score', ascending=False)

## 6. Alertes

In [ ]:
pd.DataFrame(stats.calculer_alertes(res))

## 7. Incident CR-002 : impact du seuil HITL 70 → 90 sur les patients
Le rapport complet est produit par `python scripts/simulate_incident.py`.

In [ ]:
cfg11 = governance.lire_json(config.GOV / 'versions' / 'v1.1.0.json')
rows = []
for d in demandes:
    if d['secteur'] != 'sante': continue
    r70 = pipeline.executer(d['secteur'], d['entite_id'], d['indicateurs'], cfg, random.Random(1))[0]
    r90 = pipeline.executer(d['secteur'], d['entite_id'], d['indicateurs'], cfg11, random.Random(1))[0]
    rows.append({'patient': d['entite_id'], 'score': r70['score'], 'route v1.0.0 (70)': r70['route'], 'route v1.1.0 (90)': r90['route']})
cmp = pd.DataFrame(rows)
cmp[cmp['route v1.0.0 (70)'] != cmp['route v1.1.0 (90)']]

In [ ]:
print(json.dumps(json.load(open(config.OUTPUTS / 'incident' / 'rapport_incident.json'))['etapes'], ensure_ascii=False, indent=1))